# Create Mont Blanc DEM mosaic

This notebook combines multiple 1-arc-second DEM GeoTIFF tiles covering the Mont Blanc region into a single raster file for later terrain and climate visualisation workflows.

In [ ]:
# Install required packages if running in a fresh environment
# Uncomment if needed:
# !pip install xarray rioxarray rasterio numpy pandas matplotlib h5py

import glob

import rasterio
from rasterio.merge import merge

In [ ]:
# Find all DEM tiles covering the Mont Blanc region
tif_files = glob.glob("n45_e00*_1arc_v3.tif")

print(f"Found {len(tif_files)} DEM tile(s):")
for file in tif_files:
    print(file)

In [ ]:
# Open the DEM tiles
src_files_to_mosaic = [rasterio.open(file) for file in tif_files]

# Merge tiles into one raster mosaic
mosaic, out_transform = merge(src_files_to_mosaic)

In [ ]:
# Copy metadata from the first tile and update it for the mosaic output
out_meta = src_files_to_mosaic[0].meta.copy()

out_meta.update({
    "driver": "GTiff",
    "height": mosaic.shape[1],
    "width": mosaic.shape[2],
    "transform": out_transform
})

In [ ]:
# Save the merged DEM mosaic
output_path = "montblanc_dem_mosaic.tif"

with rasterio.open(output_path, "w", **out_meta) as dest:
    dest.write(mosaic)

print(f"Saved DEM mosaic to: {output_path}")

In [ ]:
# Close open raster files
for src in src_files_to_mosaic:
    src.close()